# Exploración del dataset local

Valida la estructura y calidad de `data/listings.csv` (dataset local recuperado, esquema Inside Airbnb más reciente). **No ejecuta limpieza ni escribe salidas**: solo inspecciona los datos para confirmar que se pueden reutilizar en el pipeline de limpieza.

In [1]:
import numpy as np
import pandas as pd

RUTA = "../data/listings.csv"

df = pd.read_csv(RUTA, low_memory=False)

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

Filas: 85068
Columnas: 106


## 1. Columnas disponibles

In [2]:
for i, columna in enumerate(df.columns, start=1):
    print(f"{i}. {columna}")

1. id
2. listing_url
3. scrape_id
4. last_scraped
5. name
6. summary
7. space
8. description
9. experiences_offered
10. neighborhood_overview
11. notes
12. transit
13. access
14. interaction
15. house_rules
16. thumbnail_url
17. medium_url
18. picture_url
19. xl_picture_url
20. host_id
21. host_url
22. host_name
23. host_since
24. host_location
25. host_about
26. host_response_time
27. host_response_rate
28. host_acceptance_rate
29. host_is_superhost
30. host_thumbnail_url
31. host_picture_url
32. host_neighbourhood
33. host_listings_count
34. host_total_listings_count
35. host_verifications
36. host_has_profile_pic
37. host_identity_verified
38. street
39. neighbourhood
40. neighbourhood_cleansed
41. neighbourhood_group_cleansed
42. city
43. state
44. zipcode
45. market
46. smart_location
47. country_code
48. country
49. latitude
50. longitude
51. is_location_exact
52. property_type
53. room_type
54. accommodates
55. bathrooms
56. bedrooms
57. beds
58. bed_type
59. amenities
60. squar

## 2. Tipos de datos

In [3]:
print(df.dtypes)

id                                                int64
listing_url                                         str
scrape_id                                         int64
last_scraped                                        str
name                                                str
                                                 ...   
calculated_host_listings_count                    int64
calculated_host_listings_count_entire_homes       int64
calculated_host_listings_count_private_rooms      int64
calculated_host_listings_count_shared_rooms       int64
reviews_per_month                               float64
Length: 106, dtype: object


## 3. Valores faltantes

Resumen de columnas con datos faltantes (absoluto y porcentaje).

In [4]:
faltantes = pd.DataFrame({
    "Columna": df.columns,
    "Faltantes": df.isnull().sum().values,
    "Porcentaje": df.isnull().mean().values * 100
})

faltantes = (
    faltantes[faltantes["Faltantes"] > 0]
    .sort_values("Porcentaje", ascending=False)
    .reset_index(drop=True)
)

print("Columnas con faltantes:", len(faltantes))
display(faltantes.head(30))

Columnas con faltantes: 55


,Columna,Faltantes,Porcentaje
0,xl_picture_url,85068,100.000000
1,medium_url,85068,100.000000
2,thumbnail_url,85068,100.000000
3,host_acceptance_rate,85068,100.000000
4,neighbourhood_group_cleansed,85068,100.000000
5,jurisdiction_names,85066,99.997649
6,license,84889,99.789580
7,square_feet,84671,99.533315
8,monthly_price,80103,94.163493
9,weekly_price,78112,91.823012


## 4. Formato del precio

Verifica si `price` es texto con símbolo de moneda y prueba su conversión a numérico (sin modificar el dataset).

In [5]:
print("Tipo de price:", df["price"].dtype)
print(df["price"].head(10))

precio_num = pd.to_numeric(
    df["price"].astype(str).str.replace(r"[^\d.\-]", "", regex=True),
    errors="coerce"
)

print("\nConvertibles a numérico:", precio_num.notna().sum())
print("No convertibles:", precio_num.isna().sum())
print("Precios <= 0:", (precio_num <= 0).sum())

Tipo de price: str
0     $88.00
1     $65.00
2     $65.00
3     $79.00
4    $105.00
5    $100.00
6     $77.00
7    $300.00
8     $42.00
9    $180.00
Name: price, dtype: str

Convertibles a numérico: 85068
No convertibles: 0
Precios <= 0: 18


## 5. Columnas requeridas por el pipeline de limpieza

Confirma qué variables que usa el pipeline existen en este dataset local.

In [6]:
requeridas = [
    "id", "price", "latitude", "longitude", "accommodates",
    "bedrooms", "beds", "bathrooms", "bathrooms_text",
    "minimum_nights", "availability_365", "number_of_reviews",
    "reviews_per_month", "review_scores_rating", "amenities",
    "room_type", "property_type", "neighbourhood_cleansed",
    "host_is_superhost", "host_identity_verified", "instant_bookable",
    "host_response_rate", "host_acceptance_rate"
]

existentes = [c for c in requeridas if c in df.columns]
faltantes = [c for c in requeridas if c not in df.columns]

print("Existentes:", len(existentes), "de", len(requeridas))
print("\nFaltantes en el dataset local:", faltantes)

Existentes: 22 de 23

Faltantes en el dataset local: ['bathrooms_text']


## 6. Geográfico

Comprueba si el dataset contiene varias ciudades.

In [ ]:
for col in ["city", "market", "smart_location"]:
    if col in df.columns:
        print(f"--- {col} ---")
        print(df[col].value_counts(dropna=False).head(10))
        print()

--- city ---
city
Greater London          46869
London                  32410
London                    647
NaN                       247
Richmond                  176
Twickenham                175
Croydon                   160
Londres                   152
Kingston upon Thames      127
Harrow                    120
Name: count, dtype: int64

--- market ---
market
London                      84258
NaN                           788
Other (International)           6
Plymouth                        4
Other (Domestic)                2
La Baule and Noirmoutier        1
Oahu                            1
Paris                           1
Leeds                           1
Suffolk                         1
Name: count, dtype: int64

--- smart_location ---
smart_location
Greater London, United Kingdom          46870
London, United Kingdom                  32424
London , United Kingdom                   647
England, United Kingdom                   206
Richmond, United Kingdom                  17

## 7. IDs únicos y duplicados

In [8]:
print("Filas totales:", len(df))
print("IDs únicos:", df["id"].nunique())
print("IDs duplicados:", df["id"].duplicated().sum())
print("Filas duplicadas completas:", df.duplicated().sum())

Filas totales: 85068
IDs únicos: 85068
IDs duplicados: 0
Filas duplicadas completas: 0
